# Learned body-heading residual-ridge benchmark

## Answer first

All **36** causal candidates failed development. The best diagnostic result was
91.388° subject-balanced heading MAE and 168.610° worst p95 50/100 Hz
disagreement. It was 2.23% worse than direct device heading. Therefore no model
was frozen and all three validation subjects remain unfetched.

In [1]:
from collections import Counter
from pathlib import Path
import json
import statistics

cwd = Path.cwd().resolve()
candidates = (cwd / "research" / "pdr", cwd, cwd.parent)
research_root = next(path for path in candidates if (path / "pdr_research").is_dir())
manifest_path = research_root / "datasets" / "manifests" / "ronin-learned-heading-development-v1.json"
report = json.loads(manifest_path.read_text(encoding="utf-8"))

{
    "candidate_count": report["candidate_count"],
    "fold_count": sum(len(candidate["folds"]) for candidate in report["candidates"]),
    "eligible_candidate_count": report["eligible_candidate_count"],
    "selected_config_id": report["selected_config_id"],
    "development_decision": report["development_decision"],
    "sensor_rows_embedded": 0,
    "model_coefficients_embedded": 0,
}

{'candidate_count': 36,
 'fold_count': 144,
 'eligible_candidate_count': 0,
 'selected_config_id': None,
 'development_decision': 'stop-no-learned-candidate-survived-development',
 'sensor_rows_embedded': 0,
 'model_coefficients_embedded': 0}

## Best diagnostic candidate

In [2]:
diagnostic = next(
    candidate
    for candidate in report["candidates"]
    if candidate["config"]["config_id"] == report["best_diagnostic_config_id"]
)
headings = [
    fold["rates"][rate]["metrics"]["heading_mae_deg"]
    for fold in diagnostic["folds"]
    for rate in ("50", "100")
]
baselines = [
    fold["device_baselines"][rate]["metrics"]["heading_mae_deg"]
    for fold in diagnostic["folds"]
    for rate in ("50", "100")
]
improvement = (statistics.fmean(baselines) - statistics.fmean(headings)) / statistics.fmean(baselines)
{
    "config_id": diagnostic["config"]["config_id"],
    **diagnostic["ranking_score"],
    "device_heading_mean_mae_deg": statistics.fmean(baselines),
    "device_heading_improvement_fraction": improvement,
    "rejection_count": len(diagnostic["rejection_reasons"]),
}

{'config_id': 'lhr-w2000-a10-t0-c090',
 'worst_sequence_mean_heading_mae_deg': 93.71395518553936,
 'subject_balanced_mean_heading_mae_deg': 91.3884409551873,
 'subject_balanced_mean_turn_mae_deg': 69.61487458186036,
 'worst_p95_rate_disagreement_deg': 168.61017858346668,
 'device_heading_mean_mae_deg': 89.3941815397456,
 'device_heading_improvement_fraction': -0.022308604218889066,
 'rejection_count': 10}

### Held-out groups

In [3]:
print(f"{'group':<8} {'MAE 50':>9} {'MAE 100':>9} {'device':>9} {'rate med':>10} {'rate p95':>10}")
print("-" * 61)
for fold in diagnostic["folds"]:
    print(
        f"{fold['held_out_subject_key']:<8} "
        f"{fold['rates']['50']['metrics']['heading_mae_deg']:>9.3f} "
        f"{fold['rates']['100']['metrics']['heading_mae_deg']:>9.3f} "
        f"{fold['device_baselines']['50']['metrics']['heading_mae_deg']:>9.3f} "
        f"{fold['rate_comparison']['median_disagreement_deg']:>10.3f} "
        f"{fold['rate_comparison']['p95_disagreement_deg']:>10.3f}"
    )

group       MAE 50   MAE 100    device   rate med   rate p95
-------------------------------------------------------------
a049        88.609    91.231    73.332     81.804    168.610
a051        94.166    85.480    63.877    107.477    167.649
a052        94.577    92.851   128.651     11.641     46.685
a054        91.141    93.052    91.717      4.081     31.986


## Failure anatomy

In [4]:
reason_counts = Counter()
for candidate in report["candidates"]:
    for reason in candidate["rejection_reasons"]:
        key = reason if reason.startswith("aggregate:") else reason.split(":", 1)[1]
        reason_counts[key] += 1

dict(sorted(reason_counts.items()))

{'aggregate:device-heading-improvement': 36,
 'aggregate:turn-mae': 36,
 'aggregate:worst-heading-mae': 36,
 'median-rate-disagreement': 142,
 'p95-rate-disagreement': 143}

In [5]:
assert report["candidate_count"] == 36
assert report["eligible_candidate_count"] == 0
assert report["selected_config_id"] is None
assert report["selected_model"] is None
assert report["development_decision"] == "stop-no-learned-candidate-survived-development"
assert set(report["validation_state"].values()) == {"not-fetched"}
assert reason_counts["aggregate:worst-heading-mae"] == 36
assert reason_counts["aggregate:turn-mae"] == 36
assert reason_counts["aggregate:device-heading-improvement"] == 36
assert reason_counts["median-rate-disagreement"] == 142
assert reason_counts["p95-rate-disagreement"] == 143
print("PASS: residual-ridge family stopped; validation remains sealed.")

PASS: residual-ridge family stopped; validation remains sealed.


## Takeaway

The pipeline produced causal, covered outputs, but integrating a learned residual
rate amplified small cross-rate and cross-subject biases over multi-minute
sequences. More ridge regularization and clipping did not solve this within the
locked grid. A direct circular-state recurrent model would be a genuinely new
family; it must be preregistered before touching the same sealed validation set.

This benchmark cannot authorize product use because RoNIN remains
non-commercial research data.